## Load the required packages

In [1]:
import os
import pandas as pd
import numpy as np
import anndata
import scanpy as sc
import torch
import warnings
from pathlib import Path
from scipy.sparse import csr_matrix
from scipy.sparse import issparse
import sys
sys.path.append(f"{os.getcwd()}/moETM-main")

from moETM.train import Trainer_moETM_for_cross_prediction, Train_moETM_for_cross_prediction
from moETM.build_model import build_moETM

warnings.filterwarnings('ignore')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/moETM"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

## Select highly variable genes

In [6]:
data = anndata.concat([train_data, test_data], axis=0)
sc.pp.normalize_total(data, target_sum=1e4)
sc.pp.log1p(data)
sc.pp.highly_variable_genes(data)
hvg_index = data.var['highly_variable'].values

train_data = train_data[:, hvg_index]
test_data = test_data[:, hvg_index]
train_data, test_data

(View of AnnData object with n_obs × n_vars = 7105 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 View of AnnData object with n_obs × n_vars = 8715 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Remove cells that do not express any proteins

In [7]:
train_protein = anndata.AnnData(X=train_data.obsm["protein_expression"], obs=train_data.obs)
test_protein = anndata.AnnData(X=test_data.obsm["protein_expression"], obs=test_data.obs)

sc.pp.filter_cells(train_protein, min_genes=1)
sc.pp.filter_cells(test_protein, min_genes=1)

train_data = train_data[train_protein.obs.index, :]
test_data = test_data[test_protein.obs.index, :]

train_data.obsm["protein_expression"] = train_protein_expression.loc[train_protein.obs.index, :]
test_data.obsm["protein_expression"] = test_protein_expression.loc[test_protein.obs.index, :]
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

In [8]:
train_adata_mod1 = anndata.AnnData(X=train_data.X, obs=train_data.obs, var=train_data.var)
test_adata_mod1 = anndata.AnnData(X=test_data.X, obs=test_data.obs, var=test_data.var)
train_adata_mod2 = anndata.AnnData(X=train_data.obsm["protein_expression"])
test_adata_mod2 = anndata.AnnData(X=test_data.obsm["protein_expression"])
train_adata_mod1, train_adata_mod2, test_adata_mod1, test_adata_mod2

(AnnData object with n_obs × n_vars = 7105 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode',
 AnnData object with n_obs × n_vars = 7105 × 198,
 AnnData object with n_obs × n_vars = 8715 × 1520
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode',
 AnnData object with n_obs × n_vars = 8715 × 198)

## Normalize raw data and convert to Torch Tensors

In [9]:
X_mod1 = train_adata_mod1.X.A if issparse(train_adata_mod1.X) else np.array(train_adata_mod1.X)
X_mod2 = np.array(train_adata_mod2.X)
batch_index = np.zeros(train_adata_mod1.shape[0])

X_mod1 = X_mod1 / X_mod1.sum(1)[:, np.newaxis]
X_mod2 = X_mod2 / X_mod2.sum(1)[:, np.newaxis]

X_mod1_train_T = torch.from_numpy(X_mod1).float()
X_mod2_train_T = torch.from_numpy(X_mod2).float()
batch_index_train_T = torch.from_numpy(batch_index).to(torch.int64).cuda()
X_mod1_train_T, X_mod2_train_T

(tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0175, 0.0000,  ..., 0.0000, 0.0175, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0056, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0074, 0.0147, 0.0000]]),
 tensor([[0.0004, 0.0043, 0.0031,  ..., 0.0016, 0.0020, 0.0059],
         [0.0012, 0.0068, 0.0004,  ..., 0.0012, 0.0016, 0.0124],
         [0.0022, 0.0085, 0.0013,  ..., 0.0004, 0.0018, 0.0094],
         ...,
         [0.0030, 0.0081, 0.0025,  ..., 0.0020, 0.0015, 0.0101],
         [0.0000, 0.0115, 0.0014,  ..., 0.0014, 0.0028, 0.0083],
         [0.0000, 0.0060, 0.0037,  ..., 0.0009, 0.0023, 0.0046]]))

In [10]:
X_mod1 = test_adata_mod1.X.A if issparse(test_adata_mod1.X) else np.array(test_adata_mod1.X)
X_mod2 = np.array(test_adata_mod2.X)
batch_index = np.zeros(test_adata_mod1.shape[0])

sum1 = X_mod1.sum(1)
sum2 = X_mod2.sum(1)

X_mod1 = X_mod1 / X_mod1.sum(1)[:, np.newaxis]
X_mod2 = X_mod2 / X_mod2.sum(1)[:, np.newaxis]

X_mod1_test_T = torch.from_numpy(X_mod1).float()
X_mod2_test_T = torch.from_numpy(X_mod2).float()
batch_index_test_T = torch.from_numpy(batch_index).to(torch.int64)
X_mod1_test_T, X_mod2_test_T

(tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0044, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0022, 0.0022],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0044, 0.0088, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0076, 0.0000]]),
 tensor([[0.0003, 0.0048, 0.0102,  ..., 0.0007, 0.0014, 0.0310],
         [0.0009, 0.0089, 0.0025,  ..., 0.0006, 0.0006, 0.0031],
         [0.0023, 0.0164, 0.0028,  ..., 0.0023, 0.0019, 0.0070],
         ...,
         [0.0000, 0.0084, 0.0018,  ..., 0.0000, 0.0011, 0.0073],
         [0.0011, 0.0111, 0.0100,  ..., 0.0008, 0.0008, 0.0237],
         [0.0017, 0.0173, 0.0000,  ..., 0.0009, 0.0035, 0.0052]]))

In [11]:
del X_mod1, X_mod2, batch_index
test_mod1_sum, test_mod2_sum = sum1, sum2

## set training hyperparameters

In [12]:
num_batch = len(batch_index_train_T.unique())
input_dim_mod1 = X_mod1_train_T.shape[1]
input_dim_mod2 = X_mod2_train_T.shape[1]
train_num = X_mod1_train_T.shape[0]
num_topic = 200
emd_dim = 400
Total_epoch = 500
batch_size = 2000
Train_set = [X_mod1_train_T, X_mod2_train_T, batch_index_train_T]
Test_set = [X_mod1_test_T, X_mod2_test_T, batch_index_test_T, test_adata_mod1, test_mod1_sum, test_mod2_sum]

## Initialize model

In [13]:
encoder_mod1, encoder_mod2, decoder, optimizer = build_moETM(input_dim_mod1, input_dim_mod2, num_batch, num_topic=num_topic, emd_dim=emd_dim)
direction = 'rna_to_another'   # Or another_to_rna
trainer = Trainer_moETM_for_cross_prediction(encoder_mod1, encoder_mod2, decoder, optimizer, direction)

## Start training

In [14]:
Train_moETM_for_cross_prediction(trainer, Total_epoch, train_num, batch_size, Train_set, Test_set, output_path=base_dir)

[epoch 0 finished time 0.421950], Pearson_1=0.0537, Spearmanr_1=0.0776
[epoch 10 finished time 0.188287], Pearson_1=0.3262, Spearmanr_1=0.2845
[epoch 20 finished time 0.186035], Pearson_1=0.5262, Spearmanr_1=0.4500
[epoch 30 finished time 0.216188], Pearson_1=0.7645, Spearmanr_1=0.6624
[epoch 40 finished time 0.186563], Pearson_1=0.8096, Spearmanr_1=0.7145
[epoch 50 finished time 0.218622], Pearson_1=0.8221, Spearmanr_1=0.7306
[epoch 60 finished time 0.182389], Pearson_1=0.8197, Spearmanr_1=0.7262
[epoch 70 finished time 0.181094], Pearson_1=0.8061, Spearmanr_1=0.7081
[epoch 80 finished time 0.186007], Pearson_1=0.7879, Spearmanr_1=0.6849
[epoch 90 finished time 0.173134], Pearson_1=0.7675, Spearmanr_1=0.6606
[epoch 100 finished time 0.187413], Pearson_1=0.7600, Spearmanr_1=0.6513
[epoch 110 finished time 0.221694], Pearson_1=0.7622, Spearmanr_1=0.6546
[epoch 120 finished time 0.215654], Pearson_1=0.7760, Spearmanr_1=0.6709
[epoch 130 finished time 0.210288], Pearson_1=0.7866, Spearman

## Perform prediction on the query

In [15]:
trainer.encoder_mod1.to('cpu')
trainer.encoder_mod2.to('cpu')
trainer.decoder.to('cpu')
recon_mod1, recon_mod2 = trainer.reconstruction(X_mod1_test_T, X_mod2_test_T, batch_index_test_T)
predicted_protein_expression = pd.DataFrame(recon_mod2.detach().cpu().numpy(), index=test_adata_mod1.obs.index, columns=train_adata_mod2.var.index)
predicted_protein_expression.to_csv(f"{base_dir}/test_predicted_protein_expression (moETM).csv")
predicted_protein_expression

,ADT_4-1BBLigand(CD137L)_A0890,ADT_CD102_A0104,ADT_CD103_A0201,ADT_CD105_A0812,ADT_CD106_A0226,ADT_CD107a(LAMP-1)_A0905,ADT_CD115(CSF-1R)_A0105,ADT_CD117(c-Kit)_A0012,ADT_CD11a_A0595,ADT_CD11b_A0014,...,ADT_TCRVr3_A0210,ADT_TCRbchain_A0120,ADT_TCRr-d_A0121,ADT_TER-119-ErythroidCells_A0122,ADT_TIGIT(Vstm3)_A0848,ADT_TLR4(CD284)-MD2Complex_A0875,ADT_Tim-4_A0567,ADT_XCR1_A0568,ADT_anti-P2RY12_A0415,ADT_integrinb7_A0214
index,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGCGATCGA-1,0.000827,0.006122,0.010078,0.001225,0.000973,0.001520,0.001165,0.001339,0.023177,0.000736,...,0.002094,0.010852,0.001092,0.000192,0.003385,0.000906,0.002221,0.000920,0.001417,0.022201
AAACCCACACCTCTAC-1,0.000858,0.009253,0.001957,0.001104,0.001821,0.000968,0.000910,0.001035,0.016582,0.001346,...,0.002224,0.000978,0.001479,0.000431,0.003042,0.000937,0.002715,0.001032,0.001743,0.005404
AAACCCACACGTCGTG-1,0.001268,0.022138,0.002554,0.001393,0.001472,0.001070,0.001337,0.001164,0.011590,0.003330,...,0.003851,0.001631,0.001807,0.000451,0.005320,0.001153,0.002874,0.001378,0.002161,0.008767
AAACCCACATGTCTAG-1,0.001261,0.015163,0.001752,0.001014,0.001420,0.001409,0.000884,0.000935,0.013304,0.002015,...,0.002444,0.000951,0.001552,0.000416,0.003518,0.001146,0.002734,0.001041,0.002122,0.005069
AAACCCAGTCAAAGTA-1,0.000720,0.012610,0.001889,0.000989,0.001423,0.001020,0.000922,0.001412,0.013321,0.001669,...,0.002463,0.001182,0.001248,0.000298,0.003538,0.000742,0.002043,0.001098,0.001765,0.007299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGCATTCCTCG-1,0.000634,0.011316,0.001828,0.001102,0.002194,0.000888,0.001027,0.001287,0.011608,0.001972,...,0.002844,0.001268,0.001124,0.000244,0.003611,0.000914,0.001844,0.000836,0.002072,0.009744
TTTGTTGGTGCTCTTC-1,0.000989,0.015647,0.001669,0.000838,0.001070,0.001474,0.001231,0.001534,0.011730,0.001675,...,0.003634,0.001914,0.001495,0.000279,0.003825,0.001160,0.002727,0.001355,0.002880,0.008490
TTTGTTGTCAGCCTTC-1,0.000767,0.003897,0.001028,0.000500,0.000824,0.000792,0.000635,0.001072,0.023084,0.001163,...,0.002781,0.015133,0.000836,0.000160,0.003159,0.001102,0.001570,0.001043,0.001471,0.002704
